# SeoulEVCheck — 1주차 ML
## 서울 전기차 충전 수요 예측 (XGBoost)
담당: 오영석 · 데이터: 한국전력공사 서울 충전량(약 63만 세션)

구·충전소 단위 **일별 충전량**을 예측하고, 수요 핫스팟(배달충전 우선지역)을 도출한다.

In [ ]:
import pandas as pd, numpy as np, joblib
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import r2_score, mean_squared_error
import xgboost as xgb
%matplotlib inline
plt.rcParams['font.family']='Malgun Gothic'; plt.rcParams['axes.unicode_minus']=False

## 1. 데이터 로드

In [ ]:
df = pd.read_excel('../data/한국전력공사_서울시 전기차 충전소 충전량_20220331.xlsx')
print(df.shape)
df.head()

## 2. 전처리
주소→구 추출(도로명/지번 혼재 대응) · 날짜 파싱 · null/음수 제거 · 밀집 구간 필터

In [ ]:
def extract_gu(a):
    if not isinstance(a,str): return None
    for t in a.split():
        if t.endswith('구'): return t
    return None
df['start']=pd.to_datetime(df['충전시작시각'],errors='coerce')
df['gu']=df['주소'].map(extract_gu)
df['충전량']=pd.to_numeric(df['충전량'],errors='coerce')
n0=len(df)
df=df.dropna(subset=['start','gu','충전량']); df=df[df['충전량']>=0]
df['ym']=df['start'].dt.to_period('M'); c=df['ym'].value_counts()
df=df[df['ym'].isin(c[c>=c.max()*0.10].index)].copy()
df['date']=df['start'].dt.date; df['weekday']=df['start'].dt.weekday; df['month']=df['start'].dt.month
print('원본',n0,'-> 정제후',len(df)); print('기간',df['date'].min(),'~',df['date'].max(),'| 구',df['gu'].nunique())

## 3. EDA — 분포·충전구분·수요 상위 구

In [ ]:
print('수요 TOP5 구:
', df.groupby('gu')['충전량'].sum().sort_values(ascending=False).head().round(0))
fig,ax=plt.subplots(1,2,figsize=(12,4))
sns.histplot(df['충전량'],bins=60,ax=ax[0]); ax[0].set_title('세션 충전량 분포(우편향)')
df.groupby('충전구분')['충전량'].mean().plot(kind='bar',ax=ax[1]); ax[1].set_title('충전구분별 평균 충전량')
plt.tight_layout(); plt.show()

## 4. 집계 (구 / 충전소 단위)

In [ ]:
gu=(df.groupby(['gu','충전구분','date'],as_index=False)
      .agg(충전량=('충전량','sum'),weekday=('weekday','first'),month=('month','first')))
st=(df.groupby(['충전소명','gu','충전구분','충전기용량','date'],as_index=False)
      .agg(충전량=('충전량','sum'),weekday=('weekday','first'),month=('month','first')))
print('gu_day',gu.shape,'| station_day',st.shape)

## 5. 모델링 — 베이스라인 vs XGBoost
타깃: 일별 충전량 · 누수 특성(세션수·충전시간) 제외 · 범주형 원-핫 · log 미사용

In [ ]:
def baseline(tr,te,keys):
    gm=tr.groupby(keys,as_index=False)['충전량'].mean().rename(columns={'충전량':'p'})
    m=te.merge(gm,on=keys,how='left'); m['p']=m['p'].fillna(tr['충전량'].mean())
    return r2_score(te['충전량'],m['p'])
def train(d,cats,nums,keys,name):
    X=pd.concat([pd.get_dummies(d[cats].astype(str),dtype=np.uint8).reset_index(drop=True),
                 d[nums].reset_index(drop=True)],axis=1); y=d['충전량'].values
    Xtr,Xte,ytr,yte,dtr,dte=train_test_split(X,y,d,test_size=0.2,random_state=42)
    b=baseline(dtr,dte,keys)
    m=xgb.XGBRegressor(n_estimators=400,max_depth=7,learning_rate=0.08,subsample=0.8,
        colsample_bytree=0.8,tree_method='hist',random_state=42).fit(Xtr,ytr)
    p=m.predict(Xte); r2=r2_score(yte,p); rmse=mean_squared_error(yte,p)**0.5
    cv=cross_val_score(m,X,y,cv=KFold(5,shuffle=True,random_state=42),scoring='r2')
    print(f'[{name}] baseline R2={b:.3f} | XGB R2={r2:.3f} RMSE={rmse:.0f} | CV R2={cv.mean():.3f}')
    return m,Xte,yte,p
mg,Xg,yg,pg=train(gu,['gu','충전구분'],['weekday','month'],['gu','충전구분','weekday'],'구')
ms,Xs,ys,ps=train(st,['충전소명','gu','충전구분'],['충전기용량','weekday','month'],['충전소명','weekday'],'충전소')

## 6. 예측 vs 실제 · 특성 중요도 (구 모델)

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(11,5))
ax[0].scatter(yg,pg,s=5,alpha=0.3); ax[0].plot([0,yg.max()],[0,yg.max()],'r--')
ax[0].set_title(f'구 모델 예측vs실제 R2={r2_score(yg,pg):.2f}'); ax[0].set_xlabel('실제(kWh)'); ax[0].set_ylabel('예측(kWh)')
pd.Series(mg.feature_importances_,index=Xg.columns).sort_values().tail(12).plot(kind='barh',ax=ax[1])
ax[1].set_title('구 모델 특성 중요도 TOP12'); plt.tight_layout(); plt.show()

## 7. 결론 및 향후
- 구 모델 R²≈0.79, 충전소 모델 R²≈0.57 — **두 모델 모두 베이스라인 초과**
- 수요 핫스팟(송파·강남·마포…) → 배달충전 우선지역
- 향후: `year` 추세 특성 / 2주 DL(LSTM 시계열) / 3주 LLM(충전 어드바이저·RAG)

In [ ]:
joblib.dump(mg,'../models/xgb_gu.pkl'); joblib.dump(ms,'../models/xgb_station.pkl')
print('모델 저장 완료')